In [1]:
import sys
import os
# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from pipeline import (
    process_vocab_word
)

import json
import requests
from constants import (
    ANKI_CONNECT_URL    
)

In [2]:
# Function to send the card to Anki
def add_card_to_anki(deck_name, vocab_data):
    """
    Sends the formatted Anki card to Anki using AnkiConnect API.
    """
    note = {
        "deckName": deck_name,
        "modelName": "Basic", 
        "fields": {
            "Front": f"{vocab_data['vocab_word']}\n\n{vocab_data['example_sentence']}",
            "Back": f"{vocab_data['vocab_translation']}\n\n{vocab_data['example_sentence_translation']}",
        },
        "audio": [
            {"url": vocab_data["vocab_audio"], "filename": "vocab_word.mp3", "fields": ["Front"]},
            {"url": vocab_data["example_sentence_translation_audio"], "filename": "example_sentence.mp3", "fields": ["Back"]}
        ]
    }
    
    payload = {"action": "addNote", "version": 6, "params": {"note": note}}
    
    response = requests.post(ANKI_CONNECT_URL, json=payload).json()
    return response

In [3]:
def request(action, **params):
    return {'action': action, 'params': params, 'version': 6}

def invoke(action, **params):
    payload = request(action, **params)
    response = requests.post(ANKI_CONNECT_URL, json=payload)
    
    if not response.ok:
        raise Exception(f"Request failed with status code {response.status_code}: {response.text}")
    
    response_json = response.json()
    
    if 'error' not in response_json or 'result' not in response_json:
        raise Exception('Invalid response structure')
    if response_json['error'] is not None:
        raise Exception(response_json['error'])
    
    return response_json['result']

In [4]:
# invoke('deckNames')

In [5]:
# invoke('createDeck', deck='test1')

In [6]:
# invoke('modelNames')

In [7]:
# print my vocab mining card type
invoke("modelTemplates", modelName="Vocab Mining")

{'Card 1': {'Front': '<span style="font-size: 50px;">{{Target Word}}</span><br>\n[sound:{{Word Audio}}]',
  'Back': '{{FrontSide}}\n<hr id=answer>\n<span style="font-size: 35px;">{{Source Translation}}</span><br>\n[sound:{{Word Audio}}][sound:{{Sentence Audio}}]<br>\n<span style="font-size: 25px;">{{Target Sentence}}<br></span>\n{{Source Sentence Translation}}<br>\n'},
 'Card 2': {'Front': '<span style="font-size: 50px;">{{Source Translation}}</span>',
  'Back': '{{FrontSide}}\n\n<hr id=answer>\n<span style="font-size: 35px;">{{Target Word}}</span><br>\n[sound:{{Word Audio}}][sound:{{Sentence Audio}}]<br>\n<span style="font-size: 25px;">{{Target Sentence}}<br></span>\n{{Source Sentence Translation}}<br>\n'}}

In [8]:
invoke("modelFieldNames", modelName="Vocab Mining")

['Target Word',
 'Source Translation',
 'Target Sentence',
 'Source Sentence Translation',
 'Word Audio',
 'Sentence Audio']

In [9]:
# generate test fields
"""
{
    "vocab_word": vocab_word,
    "vocab_translation": vocab_translation,
    "example_sentence": example_sentence,
    "example_sentence_translation": example_sentence_translation,
    "vocab_audio": vocab_audio,
    "example_sentence_translation_audio": example_sentence_audio
}
"""
# todo: infer language based on input?
result = process_vocab_word("猫", target_language="Japanese")


Processing vocabulary word: 猫



KeyError: 'vocab_audio'

In [10]:
vocab_word = result["vocab_word"]
vocab_translation = result["vocab_translation"]
example_sentence = result["example_sentence"]
example_sentence_translation = result["example_sentence_translation"]
vocab_audio_filename = result["vocab_audio_filename"]
example_sentence_translation_audio_filename = result["example_sentence_translation_audio_filename"]

In [11]:

deck_name = "test1"
model_name = "Vocab Mining"
fields = {
    'Target Word': vocab_word,
    'Source Translation': vocab_translation,
    'Target Sentence': example_sentence,
    'Source Sentence Translation':example_sentence_translation,
}
audio = [
    {
        "filename": vocab_audio_filename,
        "fields": [
            "Word Audio"
        ]
    },
    {
        "filename": example_sentence_translation_audio_filename,
        "fields": [
            "Sentence Audio"
        ]
    }
]
note = {
    "deckName": deck_name,
    "modelName": model_name,
    "fields": fields,
    "audio": audio,
    "tags": ["stenchtoast"],
    "options": {
            "allowDuplicate": False,
            "duplicateScope": "deck",
            "duplicateScopeOptions": {
                "deckName": deck_name,
                "checkChildren": False,
                "checkAllModels": False
            }
    }
}

In [13]:
# add card to deck
invoke("addNote", note=note)

1741055398609